# Embedding alignment through parallel corpus

## Imports

In [1]:
!pip install -q wandb datasets transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00


In [2]:
import os
import torch
import wandb
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))

### Model and tokenizer from vocab extension wandb

In [5]:
run = wandb.init(entity="e278979-metu-middle-east-technical-university", project="bashllama-vocab-extension")
artifact = run.use_artifact("e278979-metu-middle-east-technical-university/bashllama-vocab-extension/llama2_bashkir_vocab:latest", type="model")

model_dir = artifact.download()

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260526_230308-lrq2dqig
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dauntless-hill-7
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/lrq2dqig
wandb: Downloading large artifact 'llama2_bashkir_vocab:latest', 4142.38MB. 6 files...
wandb:   6 of 6 files downloaded.  
Done. 00:00:33.8 (122.7MB/s)


In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_cache=False
)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
model = prepare_model_for_kbit_training(model)

### Lora config

we need to train only *embedding* and *lm_head*

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],      # standart for LoRA
    modules_to_save=["embed_tokens", "lm_head"],  # re-trained fully
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [10]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 498,819,072 || all params: 7,469,715,456 || trainable%: 6.6779


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


## Data - corpus -> dataset

In [11]:
# params
# data usage regime: True - subset, False - all data
USE_SUBSET = True          # true for test
SUBSET_SIZE = 10000        # numof pairs if USE_SUBSET=True (overall examples 2*SUBSET_SIZE)

In [12]:
parallel = load_dataset("AigizK/bashkir-russian-parallel-corpora", split="train", streaming=True)
# print(f"Overall pairs: {len(parallel)}")

if USE_SUBSET:
    parallel = parallel.take(SUBSET_SIZE)
    print(f"Using {SUBSET_SIZE} pairs")
else:
    print("Use all dataset iteratively")

README.md:   0%|          | 0.00/862 [00:00<?, ?B/s]

Using 10000 pairs


### Tokenizing & mixing of lang

All as mentioned in LLamaTurk

In [13]:
def tokenize_pair(example):
    ru = tokenizer(example['ru'], truncation=True, max_length=128, padding='max_length')
    ba = tokenizer(example['ba'], truncation=True, max_length=128, padding='max_length')
    return {
        'ru_input_ids': ru['input_ids'],
        'ru_attention_mask': ru['attention_mask'],
        'ba_input_ids': ba['input_ids'],
        'ba_attention_mask': ba['attention_mask']
    }



In [14]:
tokenized = parallel.map(tokenize_pair, remove_columns=['source', 'target'])


In [15]:
def example_generator():
    for ex in tokenized:
        yield {'input_ids': ex['ru_input_ids'], 'attention_mask': ex['ru_attention_mask']}
        yield {'input_ids': ex['ba_input_ids'], 'attention_mask': ex['ba_attention_mask']}

from datasets import IterableDataset
mixed_dataset = IterableDataset.from_generator(example_generator)
print("mixed dataset created")

mixed dataset created


## Model

### Freeze layers (except embeds and lm head)

In [16]:
# for param in model.parameters():
#     param.requires_grad = False

# for param in model.model.embed_tokens.parameters():
#     param.requires_grad = True
# for param in model.lm_head.parameters():
#     param.requires_grad = True

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Trainable params: {trainable:,} (≈ {trainable/1e6:.2f}M)")

In [17]:
for name, param in model.named_parameters():
    if "embed_tokens" in name or "lm_head" in name:
        print(f"{name}: requires_grad={param.requires_grad}")

base_model.model.model.embed_tokens.original_module.weight: requires_grad=False
base_model.model.model.embed_tokens.modules_to_save.default.weight: requires_grad=True
base_model.model.lm_head.original_module.weight: requires_grad=False
base_model.model.lm_head.modules_to_save.default.weight: requires_grad=True


### Training!

In [18]:
# params
MAX_STEPS = 1000 
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
GRAD_ACCUM = 4             # effecctive batch = BATCH_SIZE * GRAD_ACCUM

In [19]:
training_args = TrainingArguments(
    output_dir="./alignment_checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=50,
    save_steps=500,
    report_to="wandb",
    run_name="embedding_alignment",
    remove_unused_columns=False,
    fp16=True,
    max_steps=1000,
    num_train_epochs=1.0,
    disable_tqdm=False,
)

In [20]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [21]:
wandb.init(project="bashllama-embedding-alignment", name="alignment_run")


wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading summary, console lines 9-15
wandb: 🚀 View run dauntless-hill-7 at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/lrq2dqig
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260526_230308-lrq2dqig/logs
wandb: setting up run 5h6bkgtl
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260526_230401-5h6bkgtl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run alignment_run
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashl

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=mixed_dataset,
    data_collator=data_collator,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
50,3.083707
100,2.736373
150,2.676305
200,2.652332
250,2.563072
300,2.539003
350,2.549386
400,2.551491
450,2.504951
500,2.509642


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=1000, training_loss=2.527720863342285, metrics={'train_runtime': 10581.7255, 'train_samples_per_second': 1.512, 'train_steps_per_second': 0.095, 'total_flos': 8.5709914374144e+16, 'train_loss': 2.527720863342285, 'epoch': 1.0})

In [23]:
output_dir = "/kaggle/working/llama2_bashkir_vocab_aligned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

artifact = wandb.Artifact(
    name="llama2_bashkir_vocab_aligned",
    type="model",
    description=f"Alignment on {'subset' if USE_SUBSET else 'full'} parallel corpus"
)
artifact.add_dir(output_dir)
wandb.log_artifact(artifact)
print(f"Model saved into {output_dir}")

wandb.finish()

wandb: Adding directory to artifact (/kaggle/working/llama2_bashkir_vocab_aligned)... Done. 17.0s
wandb: uploading artifact llama2_bashkir_vocab_aligned; updating run metadata


Model saved into /kaggle/working/llama2_bashkir_vocab_aligned


wandb: uploading artifact llama2_bashkir_vocab_aligned
wandb: uploading data
wandb: 
wandb: Run history:
wandb:         train/epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
wandb:   train/global_step ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
wandb:     train/grad_norm ▄▃▅▆▅█▄▃▃▅▃▂▄▅▄▄▃▁▃▃
wandb: train/learning_rate ██▇▇▇▆▆▅▅▅▄▄▄▃▃▂▂▂▁▁
wandb:          train/loss █▅▄▄▃▃▃▃▂▂▃▂▂▂▂▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:               total_flos 8.5709914374144e+16
wandb:              train/epoch 1
wandb:        train/global_step 1000
wandb:          train/grad_norm 4.61219
wandb:      train/learning_rate 0.0
wandb:               train/loss 2.38682
wandb:               train_loss 2.52772
wandb:            train_runtime 10581.7255
wandb: train_samples_per_second 1.512
wandb:   train_steps_per_second 0.095
wandb: 
wandb: 🚀 View run alignment_run at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment/runs/5h6bkgtl
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technic